# Unit 15 / Chapter 15: Quantum Time Series and Forecasting

> **Main Learning Objective:** Learn how quantum computers can process time series (like stock prices, sensor readings, or weather data) using **quantum reservoir computing**, and forecast the next values of a simple signal.

| Section | Topic |
|---|---|
| 15.1 | Time series basics: what makes them hard |
| 15.2 | Reservoir computing (classical) in one page |
| 15.3 | Quantum reservoir computing |
| 15.4 | Forecasting a sine wave with a quantum reservoir |

---
## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(7); random.seed(7)
plt.rcParams['figure.dpi'] = 100

# Tiny quantum simulator used across all units
def ket0(n):
    s = np.zeros(2**n, dtype=complex); s[0] = 1.0
    return s
def kron_all(mats):
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out
I2 = np.eye(2, dtype=complex)
X  = np.array([[0,1],[1,0]], dtype=complex)
Y  = np.array([[0,-1j],[1j,0]], dtype=complex)
Z  = np.array([[1,0],[0,-1]], dtype=complex)
H  = (1/np.sqrt(2))*np.array([[1,1],[1,-1]], dtype=complex)
def Rx(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-1j*s],[-1j*s,c]], dtype=complex)
def Ry(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-s],[s,c]], dtype=complex)
def Rz(t): return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]], dtype=complex)
def apply_1q(gate, qubit, n):
    return kron_all([gate if i==qubit else I2 for i in range(n)])
def apply_cnot(control, target, n):
    dim = 2**n
    op = np.zeros((dim, dim), dtype=complex)
    for x in range(dim):
        bits = [(x >> (n-1-i)) & 1 for i in range(n)]
        if bits[control] == 1:
            bits[target] ^= 1
        y = 0
        for b in bits:
            y = (y<<1) | b
        op[y, x] = 1
    return op
def expZ(state, qubit, n):
    Zop = apply_1q(Z, qubit, n)
    return float(np.real(np.conj(state) @ Zop @ state))
print("Quantum simulator ready.")

---
## Course check-in

This logs that you started **Unit 15**. Enter the email you signed up with.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 15
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 15.1: Time Series Basics: What Makes Them Hard

A **time series** is an ordered sequence of measurements: temperature readings every hour, stock prices every minute, EEG voltage every millisecond. Order matters, and future values depend on past values in complicated ways.

Two classical approaches to forecasting:

* **Autoregressive (AR) models**: y_t = a1 y_{t-1} + a2 y_{t-2} + ... + noise. Simple, fast, good for stationary signals.
* **Recurrent neural nets (RNNs, LSTMs, GRUs)**: neural networks with an internal state that gets updated at each time step. Powerful but hard to train.

Both fail when the signal has:

* long-range dependencies (something at t=1000 depends on t=1),
* chaotic dynamics (small errors compound),
* high dimensionality (each time step is itself a vector).

Enter reservoir computing.

---
# Section 15.2: Reservoir Computing in One Page

**Reservoir computing (RC)** is a shortcut for training recurrent networks. The idea:

1. Build a random, fixed (untrained!) recurrent network with N nodes and complicated dynamics: the "reservoir."
2. Feed your input signal into the reservoir. The reservoir's internal state at each time step is a rich, nonlinear encoding of the input's history.
3. Train **only a linear readout** from the reservoir state to the target. Everything upstream is frozen.

Because you only train a linear layer, training is fast and stable. Because the reservoir is complex, the linear readout has plenty of features to work with.

Classical RC works well but requires a physical medium with rich dynamics. Guess what has spectacularly rich dynamics? A many-body quantum system.

---
# Section 15.3: Quantum Reservoir Computing

**Quantum Reservoir Computing (QRC)**, first proposed by Fujii and Nakajima (2017), uses a many-qubit register as the reservoir:

1. At each time step t, inject the input value x_t into the state via a small rotation.
2. Let the whole n-qubit register evolve under a fixed (untrained) random Hamiltonian for a short time.
3. Measure a fixed set of observables (say <Z_i> for every qubit) to get a feature vector.
4. Train a linear readout from those features to the target y_t.

The internal Hilbert space is 2^n dimensional, so a small quantum system offers exponentially many "features" to the readout, without needing exponentially many measurements. Recent experiments have already run QRC on 5 to 10 qubit systems and beaten classical baselines on chaotic time series.

We will simulate this with a small hand-rolled quantum reservoir.

In [ ]:
def build_random_unitary(n, seed=1):
    rng = np.random.default_rng(seed)
    A = rng.normal(size=(2**n, 2**n)) + 1j*rng.normal(size=(2**n, 2**n))
    Q, _ = np.linalg.qr(A)
    return Q

def quantum_reservoir_step(state, x, U_fixed, n):
    # inject input on qubit 0 via Ry(pi * x)
    state = apply_1q(Ry(np.pi * x), 0, n) @ state
    # evolve under fixed random unitary
    state = U_fixed @ state
    return state

def features(state, n):
    return np.array([expZ(state, i, n) for i in range(n)])

n_res = 4
U_fixed = build_random_unitary(n_res, seed=42)
state = ket0(n_res)
feat = features(state, n_res)
print("initial features:", np.round(feat, 3))

---
# Section 15.4: Forecasting a Sine Wave

We now train the linear readout. Task: given a noisy sine wave, predict the next value. Because the sine's history is a linear function of a few internal states, this is a great sanity check.

In [ ]:
T = 200
t_vals = np.linspace(0, 6*np.pi, T)
signal = 0.5*np.sin(t_vals) + 0.5

# Run the reservoir over the signal, collect features
state = ket0(n_res)
feats = []
for x in signal:
    state = quantum_reservoir_step(state, x, U_fixed, n_res)
    feats.append(features(state, n_res))
feats = np.array(feats)
print("features shape:", feats.shape)   # (T, n_res)

# Train linear readout: predict signal[t+1] from feats[t]
X_train = feats[:-1]
y_train = signal[1:]
w, *_ = np.linalg.lstsq(X_train, y_train, rcond=None)

y_pred = X_train @ w
mse = float(np.mean((y_pred - y_train)**2))
print("training MSE:", round(mse, 4))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(y_train, label="true next value")
ax.plot(y_pred,  label="quantum reservoir prediction", lw=1.2)
ax.legend(); ax.set_title("Sine wave 1-step forecast via quantum reservoir")
plt.tight_layout(); plt.show()

### Activity 15.1

Increase the reservoir size to 6 qubits and re-run the forecasting cell. Does the MSE go down? (Larger reservoir means more features for the linear readout.)

In [ ]:
# TODO: rebuild the reservoir with n_res = 6 and retrain the linear readout.
n_res_new = None
if n_res_new is not None:
    U_new  = build_random_unitary(n_res_new, seed=42)
    st     = ket0(n_res_new)
    feats2 = []
    for x in signal:
        st = quantum_reservoir_step(st, x, U_new, n_res_new)
        feats2.append(features(st, n_res_new))
    feats2 = np.array(feats2)
    w2, *_ = np.linalg.lstsq(feats2[:-1], signal[1:], rcond=None)
    mse2 = float(np.mean((feats2[:-1] @ w2 - signal[1:])**2))
    print(f"n_res = {n_res_new}, MSE = {mse2:.4f}")
else:
    print("Fill in n_res_new, then re-run.")

<details><summary>Solution</summary>

```python
n_res_new = 6
```
The MSE should drop notably because the readout has 6 nonlinear features per time step instead of 4.
</details>

### Activity 15.2

Change the input signal to a noisy signal like `signal = 0.5*np.sin(t_vals) + 0.1*np.random.randn(T)` and see how the forecast handles noise.

In [ ]:
# TODO: swap in a noisy signal and rerun the reservoir + linear readout.
noisy_signal = None
if noisy_signal is not None:
    st = ket0(n_res); f3 = []
    for x in noisy_signal:
        st = quantum_reservoir_step(st, x, U_fixed, n_res)
        f3.append(features(st, n_res))
    f3 = np.array(f3)
    w3, *_ = np.linalg.lstsq(f3[:-1], noisy_signal[1:], rcond=None)
    mse3 = float(np.mean((f3[:-1] @ w3 - noisy_signal[1:])**2))
    print("noisy signal MSE:", round(mse3, 4))
else:
    print("Fill in noisy_signal, then re-run.")

<details><summary>Solution</summary>

```python
noisy_signal = 0.5*np.sin(t_vals) + 0.1*np.random.randn(T)
```
The MSE goes up because the noise itself is unpredictable, but the sine component is still tracked.
</details>

---
## Section summary

* Time series forecasting is hard because of long-range dependencies, chaos, and high dimensionality.
* Reservoir computing freezes a rich random recurrent system and trains only a linear readout.
* Quantum reservoir computing uses the exponential Hilbert space of a many-qubit system as the reservoir.
* A 4-qubit reservoir can already fit a sine wave with tiny error, and larger reservoirs handle more complex signals.

---
## Course wrap-up

This is Unit 15, the final unit of the course. Over 15 units you built:

* A quantum simulator from scratch.
* Variational quantum classifiers, quantum kernel methods, quantum neural networks.
* Quantum algorithms like QFT, QAOA, Grover, HHL.
* Quantum error correction primitives.
* Quantum reinforcement learning, generative models, NLP, computer vision, recommenders, federated learning, and time series.

Complete this quiz and the certificate email will land in your inbox.

---
## End-of-Unit Quiz (10 multiple choice)

**Q1.** Which of the following is NOT a challenge for classical time series models?

A. Long-range dependencies
B. Chaotic dynamics
C. High-dimensional inputs
D. Integer arithmetic

**Q2.** In reservoir computing, which part is trained?

A. The reservoir dynamics
B. Only a linear readout on top of the frozen reservoir
C. Both the reservoir and the readout
D. Nothing

**Q3.** Quantum Reservoir Computing was proposed in 2017 by:

A. Feynman and Deutsch
B. Fujii and Nakajima
C. Grover and Shor
D. Kerenidis and Prakash

**Q4.** The exponential feature advantage of QRC comes from:

A. Faster classical CPUs
B. The 2^n dimensional Hilbert space of an n-qubit register
C. Special-purpose hardware
D. Bigger training sets

**Q5.** In our simulation, the reservoir was:

A. Trained gradient by gradient
B. A fixed, random unitary applied every step
C. A classical MLP
D. A recurrent LSTM

**Q6.** What features did we extract from the reservoir at each time step?

A. The full state vector
B. <Z_i> for each qubit
C. The Hamiltonian eigenvalues
D. The gate count

**Q7.** After extracting features, the readout was trained by:

A. Backpropagation through the reservoir
B. Ordinary least squares (linear regression)
C. Reinforcement learning
D. Grid search

**Q8.** Increasing the reservoir from 4 to 6 qubits usually:

A. Increases MSE
B. Decreases MSE because the readout has more nonlinear features
C. Has no effect
D. Breaks the code

**Q9.** Which is the closest classical analog to QRC?

A. Convolutional neural networks
B. Echo state networks / reservoir computing
C. K-means clustering
D. Support vector machines

**Q10.** A key practical advantage of QRC over deep RNN training is:

A. Only the linear readout is trained, so training is fast and stable
B. It needs no data
C. It uses no measurement
D. It runs only on error-corrected hardware

---
## End-of-unit submission

Fill in your ten multiple choice answers, then run this cell to submit.

In [ ]:
quiz_answers = {
    "q1":  "",   # A, B, C, or D
    "q2":  "",
    "q3":  "",
    "q4":  "",
    "q5":  "",
    "q6":  "",
    "q7":  "",
    "q8":  "",
    "q9":  "",
    "q10": ""
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 15!")